In [1]:
import sys
import os
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import pickle
from copy import copy
from math import pi, sqrt
%matplotlib qt

parent_dir = os.path.abspath("..")
if parent_dir not in sys.path:
    print(f"appending {parent_dir} to path")
    sys.path.append(parent_dir)

import src.unit_conv as conv

appending /home/artur/inzyn/quantum_decaf/view to path


## IMPORT DATA ##

In [ ]:
quantum_root = Path(parent_dir).parents[0]
fname = "./70x70x30small/all/time_evol_runs.pickle"
fname = quantum_root / fname

with open(fname, 'rb') as f:
    runs = pickle.load(f)

xr = [runs[0]["r_right"][0][0]]
xl = [runs[0]["r_left"][0][0]]
xrm = [runs[0]["r_right_max"][0][0]]
xlm = [runs[0]["r_left_max"][0][0]]

times = [0]
n_part = runs[0]["n_part"]

e_tot = [runs[0]["energies"]["total"][0]*conv.hartree_to_nK]
e_kin = [runs[0]["energies"]["kinetic"][0]*conv.hartree_to_nK]
e_ext = [runs[0]["energies"]["ext"][0]*conv.hartree_to_nK]
e_contact = [runs[0]["energies"]["contact"][0]*conv.hartree_to_nK]
e_contact_ext = [runs[0]["energies"]["contact_ext"][0]*conv.hartree_to_nK]
e_dd = [runs[0]["energies"]["dipole"][0]*conv.hartree_to_nK]

for i, run in enumerate(runs):
    r_right = run["r_right"][0][1:]
    r_left = run["r_left"][0][1:]
    rm_right = run["r_right_max"][0][1:]
    rm_left = run["r_left_max"][0][1:]

    tot = run["energies"]["total"][1:]*conv.hartree_to_nK
    kin = run["energies"]["kinetic"][1:]*conv.hartree_to_nK
    ext = run["energies"]["ext"][1:]*conv.hartree_to_nK
    contact = run["energies"]["contact"][1:]*conv.hartree_to_nK
    contact_ext = run["energies"]["contact_ext"][1:]*conv.hartree_to_nK
    dd = run["energies"]["dipole"][1:]*conv.hartree_to_nK

    n_steps = run["n_steps"]
    dt = run["timestep"]*conv.jat_to_ms
    timestamps = np.arange(1, n_steps + 1)*dt
    times.extend(timestamps + copy(times[-1]))

    xr.extend(r_right.copy())
    xl.extend(r_left.copy())
    xrm.extend(rm_right.copy())
    xlm.extend(rm_left.copy())

    e_tot.extend(tot.copy())
    e_kin.extend(kin.copy())
    e_ext.extend(ext.copy())
    e_contact.extend(contact.copy())
    e_contact_ext.extend(contact_ext.copy())
    e_dd.extend(dd.copy())

xr = np.array(xr)
xl = np.array(xl)
xrm = np.array(xrm)
xlm = np.array(xlm)

e_tot = np.array(e_tot)
e_kin = np.array(e_kin)
e_ext = np.array(e_ext)
e_contact = np.array(e_contact)
e_contact_ext = np.array(e_contact_ext)
e_dd = np.array(e_dd)

times = np.array(times)

## FIND KINETIC ENERGY CLASSICALLY ## (PER PARTICLE)

In [35]:
m = 163.929*1822.888353
dist = xr - xl
v_rel = np.empty_like(dist)
v_rel[0] = (dist[1] - dist[0])/(times[1] - times[0])
v_rel[-1] = (dist[-1] - dist[-2])/(times[-1] - times[-2])

p1 = slice(2, None, 1)
m1 = slice(0, -2, 1)
zero = slice(1, -1, 1)
v_rel[zero] = (dist[p1] - dist[m1])/2/(times[p1] - times[zero])

E_k = 0.25*m*v_rel**2*(conv.mum_to_atm*conv.jat_to_ms)**2*conv.hartree_to_nK

In [37]:
fig, ax = plt.subplots()
ax.plot(times, e_kin/n_part - e_kin[0]/n_part, label=r"$\Delta$ quantum kinetic")
ax.plot(times, E_k, label="classical kinetic", color="orange")
plt.legend()

In [38]:
fig, ax = plt.subplots()
ax.plot(times, e_kin/n_part - e_kin[0]/n_part - E_k, label=r"$\Delta$ quantum - classical kinetic")
plt.legend()

## CLASSICAL CALCULATION OF DIPOLE DIPOLE INTERACTION FOR TWO STICKS ##

In [40]:
def dd_classic(d, h):
    add = 131
    m = 163.929*1822.888353
    rho = (n_part/2/h)
    cdd = 12*pi*add/m*conv.hartree_to_nK/conv.mum_to_atm**3
    return cdd/4/pi*rho**2/d*2*(1 - 1/sqrt((h/d)**2 + 1))

classical and quamtum in time

In [75]:
vdd_classic = np.empty_like(dist)
for i in range(dist.shape[0]):
    vdd_classic[i] = dd_classic(dist[i], 2)

fig, ax = plt.subplots()
ax.plot(times, e_dd/n_part - e_dd[0]/n_part, label=r"$\Delta$ edd")
ax.plot(times, vdd_classic/n_part - vdd_classic[0]/n_part, label=r"$\Delta$ classic vdd")
plt.legend()

E_dd of R

In [76]:
fig, ax = plt.subplots()

ax.plot(dist, e_dd/n_part - e_dd[0]/n_part, "r.", label="quantum")
ax.plot(dist, vdd_classic/n_part - vdd_classic[0]/n_part, "b.", label="classic")
plt.legend()

In [77]:
fig, ax = plt.subplots()

dd_diff = e_dd/n_part - e_dd[0]/n_part - vdd_classic/n_part + vdd_classic/n_part
t_diff = e_kin/n_part - e_kin[0]/n_part - E_k
ax.plot(times, -dd_diff , label="V_dd: classic - quantum")
ax.plot(times, t_diff, label=r"$\Delta$ quantum - classical kinetic", color="orange")
plt.legend()

In [78]:
fix, ax = plt.subplots(1,2)
ax = ax.ravel()

ax[0].plot(times, e_kin/n_part + vdd_classic/n_part)
ax[1].plot(dist, e_kin/n_part + vdd_classic/n_part, 'r.')

In [66]:
fig, ax = plt.subplots()
ax.plot(dist, e_kin/n_part + e_dd/n_part - e_kin[0]/n_part - e_dd[0]/n_part, 'r.')
ax.plot(dist, E_k/n_part + vdd_classic/n_part - e_dd[0]/n_part, 'b.')